> ## SOLUTION / ANSWER KEY &mdash; Lab 6.6
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-06-tool-routing.ipynb`](../lab-06-tool-routing.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 6.6 &mdash; Routing an Action to a Tool (safely)

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- Look a tool up by name in a registry and run it with .invoke()
- Handle an unknown tool and a failing tool without crashing
- Then let the real agent do the same routing over the same tools

> **How this lab works (near-real):** you have a local Ollama running `llama3.1:8b`. Read the **Concept**, fill the real `___` blanks in **Build it** (real tool bodies, real prompts, the real `create_agent` call), then **Run it for real** &mdash; a real LLM drives a real agent over real tools &mdash; and **read the trace** to see exactly what it did. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`, `langgraph`) and a **real local model** (`ollama run llama3.1:8b`, pinned to `http://127.0.0.1:11434`). If Ollama is down, the run cells print how to start it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* aborts the whole agent run (you'll see exactly this in Lab 11).

**Reference:** [Module 6 slides &mdash; LangChain's core building blocks](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))   # SERPER_API_KEY, WOLFRAM_ALPHA_APPID, GROQ/OPENAI keys

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-06-06")
os.makedirs(WORK, exist_ok=True)

def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening. If it's down, start it with:  ollama run llama3.1:8b"""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

from langchain_ollama import ChatOllama
# Day-3 provider: a REAL local model. Pin the host -- plain 'localhost' can give 'No route to host'.
llm = ChatOllama(model="llama3.1:8b", temperature=0, base_url="http://127.0.0.1:11434")

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if ollama_up():
    print("Ollama reachable at 127.0.0.1:11434 | model:", llm.model, "| WORK:", WORK)
else:
    print("Ollama NOT reachable -- start it with:  ollama run llama3.1:8b")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Once the model picks an **action name**, the framework must **route** it to the matching tool, run it
via `.invoke()`, and wrap the result as an **observation**. `create_agent` does this internally &mdash;
here you build the same safe dispatch by hand so you can debug it. Models hallucinate tool names, so
routing must fail **safely**: an unknown or failing tool returns a message, not a crash. (This is genuine
rule-based plumbing, not an LLM stand-in.)

In [ ]:
from langchain_core.tools import tool

@tool
def weather(city: str) -> str:
    """Get the current weather for a city."""
    return "sunny, 24C"

print("a tool has .name and .invoke:", weather.name, "->", weather.invoke("tokyo"))

## Build it
Build a registry of real tools, then implement `dispatch`: find the tool, run it via `.invoke()`, and
handle the unknown/failing cases without crashing.

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)

from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    """Do exact arithmetic on an expression."""
    try:
        return str(safe_calc(expression))
    except Exception:
        return "error: not a numeric expression"

@tool
def lookup(key: str) -> str:
    """Look up a known fact by key."""
    facts = {"capital of france": "Paris", "population of metropolis": "120000"}
    return facts.get(key.lower().strip(), "unknown")

REGISTRY = {t.name: t for t in [calculator, lookup]}

def dispatch(registry, name, arg):
    tool = registry.get(name)
    if tool is None:
        return {"ok": False, "observation": "unknown tool: " + name}
    try:
        return {"ok": True, "observation": tool.invoke(arg)}
    except Exception as e:
        return {"ok": False, "observation": "tool error: " + type(e).__name__}

try:
    print(dispatch(REGISTRY, "calculator", "10/2"))
    print(dispatch(REGISTRY, "lookup", "capital of france"))
    print(dispatch(REGISTRY, "no_such_tool", "x"))    # hallucinated name -> handled, no crash
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Now hand the same tools to a real agent and let *it* do the routing. Compare the trace to your by-hand dispatch.

_This calls the real `llama3.1:8b` via Ollama. If Ollama is down the cell prints how to start it instead of crashing._

In [ ]:
if not ollama_up():
    print("Ollama not reachable -- start `ollama run llama3.1:8b`, then re-run this cell.")
else:
    try:
        from langchain.agents import create_agent
        agent = create_agent(llm, tools=list(REGISTRY.values()))
        result = agent.invoke({"messages": [("user", "What is 10 divided by 2?")]},
                              config={"recursion_limit": 6})
        print_trace(result)
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- Your `dispatch` and the agent's internal routing do the **same job**: name -> tool -> observation.
- The agent's version never crashes on a bad tool name because tools return strings &mdash; the discipline you enforced by hand.
- Splitting *deciding* (the model) from *running* (the executor) is what makes agents debuggable.

## Your turn (open task &mdash; no grader)
Add a third tool to `REGISTRY`, rebuild the agent with `list(REGISTRY.values())`, and ask a question that
routes to it. **What good looks like:** the trace shows the agent routing to your new tool by name, and an
intentionally wrong tool name (in `dispatch`) still returns a clean "unknown tool" message.

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
from langchain_core.tools import tool
from langchain.agents import create_agent

@tool
def weather(city: str) -> str:
    """Get the current weather for a city."""
    return city.title() + ": sunny, 24C"

REGISTRY["weather"] = weather
print("by-hand dispatch:", dispatch(REGISTRY, "weather", "tokyo"))
print("bad tool name   :", dispatch(REGISTRY, "nope", "x"))   # clean 'unknown tool' message, no crash
agent2 = create_agent(llm, tools=list(REGISTRY.values()))
if ollama_up():
    print_trace(agent2.invoke({"messages": [("user", "What's the weather in Tokyo?")]},
                              config={"recursion_limit": 6}))
else:
    print("(start Ollama: ollama run llama3.1:8b)")

---
### Key takeaway
Routing turns a chosen action into a real observation -- safely. You built the dispatch create_agent does for you, and saw both survive a bad tool name.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>